# 01 — Análise exploratória de reservas

**CRISP-DM fase 2.3 — exploração dos dados** · Turismo e Hotelaria

---

Este notebook explora as reservas de hotéis urbanos e de resort usando a base bruta
`hotel_bookings.csv` e o dicionário gerado no notebook `00-dicionario-dados.ipynb`.
A análise cobre perfil, distribuições, cancelamentos, sazonalidade, dados ausentes,
valores extremos e associações entre variáveis.

Os resultados são descritivos. `is_canceled` é o desfecho de interesse; campos de
status final, como `reservation_status` e `reservation_status_date`, ocorrem depois
da reserva e não devem ser usados como preditores de cancelamento.

In [1]:
# Importações e caminhos do projeto
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARQUIVO_BRUTO = RAIZ / "data" / "raw" / "hotel_bookings.csv"
ARQUIVO_DICIONARIO = RAIZ / "data" / "processed" / "dicionario.csv"
FIGURAS = RAIZ / "docs" / "imagens" / "eda"
TABELAS = RAIZ / "reports" / "tables" / "eda-exploratoria"
FIGURAS.mkdir(parents=True, exist_ok=True)
TABELAS.mkdir(parents=True, exist_ok=True)

TITULO = "ANÁLISE DE RESERVAS DE HOTEL"
TOPO = 10
PALETA = sns.color_palette("colorblind")
BARRA = PALETA[0]
DESTAQUE = PALETA[3]


def salvar(fig, nome, dados=None):
    """Salva figura e, quando fornecida, sua tabela de origem."""
    fig.savefig(FIGURAS / f"{nome}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    if dados is not None:
        dados.to_csv(TABELAS / f"{nome}.csv", index=False, sep=";", encoding="utf-8")


print(f"Projeto em {RAIZ}")

Projeto em /home/luan/Área de trabalho/projeto-hotel-clientes/projeto-exemplo-mercado-imobiliario-main


## Carregando a base de dados

O notebook lê a base bruta em `data/raw/hotel_bookings.csv` e o dicionário em
`data/processed/dicionario.csv`. O dicionário documenta as 32 colunas originais;
esta análise também deriva `arrival_date`, `total_nights` e `total_guests` para
explorar calendário e dimensão da estadia.

In [2]:
# Leitura da base e do dicionário produzido na etapa anterior
dicionario = pd.read_csv(ARQUIVO_DICIONARIO, sep=";")
bruto = pd.read_csv(
    ARQUIVO_BRUTO,
    low_memory=False,
    na_values=["NULL"],
    dtype={"agent": "string", "company": "string"},
)

faltantes_no_csv = sorted(set(dicionario["variavel"]) - set(bruto.columns))
nao_documentadas = sorted(set(bruto.columns) - set(dicionario["variavel"]))
if faltantes_no_csv or nao_documentadas:
    raise ValueError(
        f"Dicionário e CSV não correspondem. Ausentes no CSV: {faltantes_no_csv}; "
        f"sem descrição: {nao_documentadas}"
    )

bruto["reservation_status_date"] = pd.to_datetime(bruto["reservation_status_date"], errors="coerce")
df = bruto.copy()
meses = pd.to_datetime(df["arrival_date_month"], format="%B", errors="coerce").dt.month
df["arrival_date"] = pd.to_datetime(
    {
        "year": df["arrival_date_year"],
        "month": meses,
        "day": df["arrival_date_day_of_month"],
    },
    errors="coerce",
)
df["arrival_month"] = df["arrival_date"].dt.to_period("M").dt.to_timestamp()
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df[["adults", "children", "babies"]].fillna(0).sum(axis=1)

colunas_derivadas = pd.DataFrame(
    [
        {
            "variavel": "arrival_date",
            "descricao": "Data prevista de chegada, recomposta de ano, mês e dia",
            "valores": [
                df["arrival_date"].min().date().isoformat(),
                df["arrival_date"].max().date().isoformat(),
            ],
            "Tipo": "Data",
        },
        {
            "variavel": "arrival_month",
            "descricao": "Mês de chegada, truncado ao primeiro dia do mês para séries temporais",
            "valores": [
                df["arrival_month"].min().date().isoformat(),
                df["arrival_month"].max().date().isoformat(),
            ],
            "Tipo": "Data",
        },
        {
            "variavel": "total_nights",
            "descricao": "Total de noites da estadia, somando noites de semana e fim de semana",
            "valores": [int(df["total_nights"].min()), int(df["total_nights"].max())],
            "Tipo": "Discreta",
        },
        {
            "variavel": "total_guests",
            "descricao": "Total de hóspedes informados na reserva: adultos, crianças e bebês",
            "valores": [int(df["total_guests"].min()), int(df["total_guests"].max())],
            "Tipo": "Discreta",
        },
    ]
)
dicionario = pd.concat([dicionario, colunas_derivadas], ignore_index=True)

print(f"{df.shape[0]:,} reservas × {bruto.shape[1]} colunas originais")
print(f"Datas de status válidas: {df['reservation_status_date'].notna().sum():,}")
df.head()

119,390 reservas × 32 colunas originais
Datas de status válidas: 119,390


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,arrival_month,total_nights,total_guests
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,2015-07-01,0,2.0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,2015-07-01,0,2.0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,2015-07-01,1,1.0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,2015-07-01,1,1.0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01,2015-07-01,2,2.0


## Recorte e variáveis derivadas

A análise preserva todas as colunas originais. Para facilitar as comparações, a etapa de
carga acrescenta a data de chegada, o mês de chegada, o total de noites e o total de
hóspedes. O dicionário continua sendo a fonte dos tipos das colunas originais; as quatro
variáveis derivadas recebem tipos e descrições explícitos.

`reservation_status` e `reservation_status_date` são mantidas para descrição do desfecho,
mas são campos posteriores à reserva e devem ser excluídas de qualquer modelo que tente
prever `is_canceled`.

In [3]:
# Grupos semânticos orientados pelo dicionário
tipos = dicionario.set_index("variavel")["Tipo"].to_dict()
nominal_columns = [col for col, tipo in tipos.items() if tipo == "Nominal"]
binary_columns = [col for col, tipo in tipos.items() if tipo == "Binária"]
discrete_columns = [col for col, tipo in tipos.items() if tipo == "Discreta"]
continuous_columns = [col for col, tipo in tipos.items() if tipo == "Contínua"]
data_columns = [col for col, tipo in tipos.items() if tipo == "Data"]
numericas = discrete_columns + continuous_columns
categoricas = nominal_columns + binary_columns

print(f"{len(nominal_columns)} nominais · {len(binary_columns)} binárias")
print(
    f"{len(discrete_columns)} discretas · {len(continuous_columns)} contínuas"
    f" · {len(data_columns)} temporais"
)

13 nominais · 2 binárias
17 discretas · 1 contínuas · 3 temporais


Cada grupo de tipo semântico orienta sua análise: frequências para categorias, medidas de
posição e dispersão para números, e agregações por mês para as datas. Os indicadores
binários recebem atenção especial por representar cancelamento e recorrência.

In [4]:
# Conferindo os grupos semânticos e o desfecho
resumo_tipos = dicionario["Tipo"].value_counts().rename_axis("tipo").to_frame("variaveis")
print("Variáveis no dicionário:", len(dicionario))
print("Taxa geral de cancelamento:", f"{df['is_canceled'].mean():.1%}")
resumo_tipos

Variáveis no dicionário: 36
Taxa geral de cancelamento: 37.0%


,variaveis
tipo,
Discreta,17
Nominal,13
Data,3
Binária,2
Contínua,1


# 1ª etapa · Distribuição dos atributos

O panorama resume dimensões, tipos, duplicidades e ausências da base. As linhas repetidas
são reportadas, mas não removidas: sem um identificador único de reserva, não é possível
assumir que dois registros idênticos sejam o mesmo evento.

In [5]:
def perfilar(dados: pd.DataFrame) -> dict:
    """Resumo de dimensões, tipos, duplicidades e ausência de valores."""
    quantitativas = dados.select_dtypes(include="number").columns
    celulas = dados.shape[0] * dados.shape[1]
    ausentes = int(dados.isna().sum().sum())
    return {
        "Reservas": dados.shape[0],
        "Variáveis": dados.shape[1],
        "Numéricas": len(quantitativas),
        "Categóricas ou temporais": dados.shape[1] - len(quantitativas),
        "Linhas duplicadas exatas": int(dados.duplicated().sum()),
        "Células ausentes": ausentes,
        "% de células ausentes": round(100 * ausentes / celulas, 2),
    }


perfil_inicial = perfilar(df)
pd.Series(perfil_inicial, name=TITULO).to_frame()

,ANÁLISE DE RESERVAS DE HOTEL
Reservas,119390.00
Variáveis,36.00
Numéricas,20.00
Categóricas ou temporais,16.00
Linhas duplicadas exatas,31994.00
Células ausentes,129425.00
% de células ausentes,3.01


## Registros repetidos

A base não apresenta uma chave de reserva. Por isso, duplicatas exatas são contabilizadas
para avaliação, mas permanecem na análise: excluí-las sem regra de negócio poderia remover
reservas legítimas com atributos iguais.

In [6]:
duplicados = int(df.duplicated().sum())
resumo_duplicados = pd.DataFrame(
    [
        {
            "reservas": len(df),
            "duplicatas_exatas": duplicados,
            "percentual": round(100 * duplicados / len(df), 2),
        }
    ]
)
resumo_duplicados.to_csv(TABELAS / "duplicatas.csv", index=False, sep=";", encoding="utf-8")
print(
    f"{duplicados:,} linhas idênticas"
    f" ({resumo_duplicados.loc[0, 'percentual']:.2f}%); nenhuma foi removida."
)
resumo_duplicados

31,994 linhas idênticas (26.80%); nenhuma foi removida.


,reservas,duplicatas_exatas,percentual
0,119390,31994,26.8


### Perfil coluna a coluna

O perfil reúne cardinalidade, valores ausentes e, para números, mínimo, mediana, máximo,
assimetria e curtose. Para datas, registra os limites do período observado.

In [7]:
def descrever(dados: pd.DataFrame, metadados: pd.DataFrame) -> pd.DataFrame:
    """Perfil de cardinalidade, ausência e estatísticas simples por variável."""
    linhas = []
    for meta in metadados.itertuples(index=False):
        coluna = dados[meta.variavel]
        linha = {
            "variavel": meta.variavel,
            "tipo": meta.Tipo,
            "distintos": int(coluna.nunique()),
            "ausentes": int(coluna.isna().sum()),
            "percentual_ausente": round(100 * coluna.isna().mean(), 2),
            "minimo": None,
            "mediana": None,
            "maximo": None,
            "assimetria": None,
            "curtose": None,
        }
        if pd.api.types.is_numeric_dtype(coluna):
            linha.update(
                minimo=coluna.min(),
                mediana=coluna.median(),
                maximo=coluna.max(),
                assimetria=round(coluna.skew(), 2),
                curtose=round(coluna.kurtosis(), 2),
            )
        elif pd.api.types.is_datetime64_any_dtype(coluna):
            linha.update(minimo=coluna.min(), maximo=coluna.max())
        linhas.append(linha)
    return pd.DataFrame(linhas)


perfil_colunas = descrever(df, dicionario)
perfil_colunas.to_csv(TABELAS / "perfil-das-colunas.csv", index=False, sep=";", encoding="utf-8")
perfil_colunas

,variavel,tipo,distintos,ausentes,percentual_ausente,minimo,mediana,maximo,assimetria,curtose
0,hotel,Nominal,2,0,0.00,None,NaN,None,NaN,NaN
1,is_canceled,Binária,2,0,0.00,0,0.000,1,0.54,-1.71
2,lead_time,Discreta,479,0,0.00,0,69.000,737,1.35,1.70
3,arrival_date_year,Discreta,3,0,0.00,2015,2016.000,2017,-0.23,-0.99
4,arrival_date_month,Nominal,12,0,0.00,None,NaN,None,NaN,NaN
5,arrival_date_week_number,Discreta,53,0,0.00,1,28.000,53,-0.01,-0.99
6,arrival_date_day_of_month,Discreta,31,0,0.00,1,16.000,31,-0.00,-1.19
7,stays_in_weekend_nights,Discreta,17,0,0.00,0,1.000,19,1.38,7.17
8,stays_in_week_nights,Discreta,35,0,0.00,0,2.000,50,2.86,24.28
9,adults,Discreta,14,0,0.00,0,2.000,55,18.32,1352.12


# 2ª etapa · Análise univariada

As distribuições categóricas são resumidas por frequência; as numéricas, por histogramas,
quartis e medidas de assimetria. Para campos com muitas categorias, mostramos apenas as
mais frequentes para manter os gráficos legíveis.

In [8]:
for coluna in nominal_columns:
    serie = df[coluna].fillna("Ausente").astype(str)
    contagem = serie.value_counts().head(TOPO).sort_values()
    tabela = contagem.rename_axis(coluna).reset_index(name="reservas")

    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(contagem))))
    sns.barplot(data=tabela, x="reservas", y=coluna, color=BARRA, ax=ax)
    ax.set(title=f"Categorias mais frequentes: {coluna}", xlabel="Reservas", ylabel="")
    fig.tight_layout()
    salvar(fig, f"univariada-nominal-{coluna}", tabela)

print(f"{len(nominal_columns)} variáveis nominais resumidas")

13 variáveis nominais resumidas


## Variáveis numéricas

Histogramas e diagramas de caixa ajudam a comparar concentração, assimetria e extremos nas
contagens, no prazo de antecedência e na tarifa diária. Valores extremos são sinalizados,
não removidos automaticamente.

In [9]:
resumos_numericos = []
for coluna in numericas:
    serie = df[coluna].dropna()
    resumo = {
        "variavel": coluna,
        "minimo": serie.min(),
        "q1": serie.quantile(0.25),
        "mediana": serie.median(),
        "q3": serie.quantile(0.75),
        "maximo": serie.max(),
        "media": serie.mean(),
        "desvio_padrao": serie.std(),
        "assimetria": serie.skew(),
    }
    resumos_numericos.append(resumo)

    fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(serie, bins=30, color=BARRA, ax=eixos[0])
    eixos[0].axvline(serie.median(), color=DESTAQUE, linestyle="--", label="Mediana")
    eixos[0].set(title=f"Distribuição: {coluna}", xlabel=coluna)
    eixos[0].legend()
    sns.boxplot(x=serie, color=BARRA, ax=eixos[1])
    eixos[1].set(title="Quartis e extremos", xlabel=coluna)
    fig.tight_layout()
    salvar(fig, f"univariada-numerica-{coluna}", pd.DataFrame([resumo]))

resumo_numericas = pd.DataFrame(resumos_numericos).sort_values("assimetria", ascending=False)
resumo_numericas.to_csv(TABELAS / "resumo-numericas.csv", index=False, sep=";", encoding="utf-8")
resumo_numericas

,variavel,minimo,q1,mediana,q3,maximo,media,desvio_padrao,assimetria
8,babies,0.00,0.00,0.000,0.0,10.0,0.007949,0.097436,24.646545
9,previous_cancellations,0.00,0.00,0.000,0.0,26.0,0.087118,0.844336,24.458049
10,previous_bookings_not_canceled,0.00,0.00,0.000,0.0,72.0,0.137097,1.497437,23.539800
6,adults,0.00,2.00,2.000,2.0,55.0,1.856403,0.579261,18.317805
12,days_in_waiting_list,0.00,0.00,0.000,0.0,391.0,2.321149,17.594721,11.944353
17,adr,-6.38,69.29,94.575,126.0,5400.0,101.831122,50.535790,10.530214
16,total_guests,0.00,2.00,2.000,2.0,55.0,1.968239,0.722394,10.154167
11,booking_changes,0.00,0.00,0.000,0.0,21.0,0.221124,0.652306,6.000270
13,required_car_parking_spaces,0.00,0.00,0.000,0.0,8.0,0.062518,0.245291,4.163233
7,children,0.00,0.00,0.000,0.0,10.0,0.103890,0.398561,4.112590


## Cancelamento e recorrência

`is_canceled` e `is_repeated_guest` são indicadores binários. A comparação da taxa de
cancelamento por tipo de hotel e por canal/segmento é descritiva; não implica causalidade.
Os campos de status final não entram como explicadores, pois são definidos após o resultado.

In [10]:
def taxa_cancelamento_por(coluna: str) -> pd.DataFrame:
    """Contagem e taxa de cancelamento por categoria, incluindo valores ausentes."""
    resultado = (
        df.assign(_categoria=df[coluna].fillna("Ausente").astype(str))
        .groupby("_categoria", observed=True)["is_canceled"]
        .agg(reservas="size", cancelamentos="sum", taxa_cancelamento="mean")
        .reset_index()
        .rename(columns={"_categoria": coluna})
        .sort_values("taxa_cancelamento", ascending=False)
    )
    return resultado


por_hotel = taxa_cancelamento_por("hotel")
por_segmento = taxa_cancelamento_por("market_segment")
por_deposito = taxa_cancelamento_por("deposit_type")

fig, eixos = plt.subplots(1, 3, figsize=(16, 5))
for ax, tabela, coluna in [
    (eixos[0], por_hotel, "hotel"),
    (eixos[1], por_segmento.head(TOPO), "market_segment"),
    (eixos[2], por_deposito, "deposit_type"),
]:
    sns.barplot(data=tabela, x="taxa_cancelamento", y=coluna, color=BARRA, ax=ax)
    ax.set(title=f"Por {coluna}", xlabel="Taxa de cancelamento", ylabel="")
fig.tight_layout()
salvar(fig, "cancelamento-por-categoria")

por_segmento.to_csv(
    TABELAS / "cancelamento-por-segmento.csv", index=False, sep=";", encoding="utf-8"
)
por_hotel, por_segmento, por_deposito

(          hotel  reservas  cancelamentos  taxa_cancelamento
 0    City Hotel     79330          33102           0.417270
 1  Resort Hotel     40060          11122           0.277634,
   market_segment  reservas  cancelamentos  taxa_cancelamento
 7      Undefined         2              2           1.000000
 4         Groups     19811          12097           0.610620
 6      Online TA     56477          20739           0.367211
 5  Offline TA/TO     24219           8311           0.343160
 0       Aviation       237             52           0.219409
 2      Corporate      5295            992           0.187347
 3         Direct     12606           1934           0.153419
 1  Complementary       743             97           0.130552,
   deposit_type  reservas  cancelamentos  taxa_cancelamento
 1   Non Refund     14587          14494           0.993624
 0   No Deposit    104641          29694           0.283770
 2   Refundable       162             36           0.222222)

## Sazonalidade das chegadas

As datas previstas de chegada foram recompostas a partir de ano, mês e dia. A série mensal
mostra volume de reservas e taxa de cancelamento por tipo de hotel; `reservation_status_date`
não é usada como data de chegada, pois registra uma atualização posterior do estado.

In [11]:
mensal_hotel = (
    df.groupby(["arrival_month", "hotel"], observed=True)
    .agg(
        reservas=("hotel", "size"),
        cancelamentos=("is_canceled", "sum"),
        taxa_cancelamento=("is_canceled", "mean"),
    )
    .reset_index()
)

fig, eixos = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(
    data=mensal_hotel, x="arrival_month", y="reservas", hue="hotel", marker="o", ax=eixos[0]
)
eixos[0].set(title="Reservas por mês de chegada", xlabel="", ylabel="Reservas")
sns.lineplot(
    data=mensal_hotel,
    x="arrival_month",
    y="taxa_cancelamento",
    hue="hotel",
    marker="o",
    ax=eixos[1],
)
eixos[1].set(title="Taxa de cancelamento por mês de chegada", xlabel="Mês", ylabel="Taxa")
fig.tight_layout()
salvar(fig, "serie-mensal-chegadas", mensal_hotel)
mensal_hotel.tail(10)

,arrival_month,hotel,reservas,cancelamentos,taxa_cancelamento
42,2017-04-01,City Hotel,3919,1926,0.491452
43,2017-04-01,Resort Hotel,1742,537,0.308266
44,2017-05-01,City Hotel,4556,2217,0.486611
45,2017-05-01,Resort Hotel,1757,545,0.310188
46,2017-06-01,City Hotel,3971,1808,0.455301
47,2017-06-01,Resort Hotel,1676,631,0.376492
48,2017-07-01,City Hotel,3559,1324,0.372015
49,2017-07-01,Resort Hotel,1754,660,0.376283
50,2017-08-01,City Hotel,3125,1123,0.359360
51,2017-08-01,Resort Hotel,1800,693,0.385000


# 3ª etapa · Análise bivariada

Comparamos distribuições de algumas variáveis numéricas entre reservas canceladas e não
canceladas. O resultado é descritivo e não inclui `reservation_status` nem sua data, que
revelam o desfecho depois que a reserva foi feita.

In [12]:
COLUNAS_COM_CANCELAMENTO = ["lead_time", "total_nights", "total_guests", "adr"]
comparacao_cancelamento = df.groupby("is_canceled")[COLUNAS_COM_CANCELAMENTO].median().T
comparacao_cancelamento.columns = ["não cancelada", "cancelada"]
comparacao_cancelamento.to_csv(TABELAS / "medianas-por-cancelamento.csv", sep=";", encoding="utf-8")

fig, eixos = plt.subplots(2, 2, figsize=(13, 9))
for ax, coluna in zip(eixos.flat, COLUNAS_COM_CANCELAMENTO):
    sns.boxplot(data=df, x="is_canceled", y=coluna, showfliers=False, color=BARRA, ax=ax)
    ax.set(title=f"{coluna} por resultado", xlabel="Cancelada (0 = não, 1 = sim)", ylabel=coluna)
fig.tight_layout()
salvar(fig, "numericas-por-cancelamento")
comparacao_cancelamento

,não cancelada,cancelada
lead_time,45.0,113.0
total_nights,3.0,3.0
total_guests,2.0,2.0
adr,92.5,96.2


## Evolução mensal por segmento

A composição dos segmentos pode mudar ao longo do calendário. Esta comparação usa apenas os
cinco segmentos com mais reservas e reporta, por mês de chegada, volume e taxa de
cancelamento.

In [13]:
segmentos_top = df["market_segment"].value_counts().head(5).index
mensal_segmento = (
    df[df["market_segment"].isin(segmentos_top)]
    .groupby(["arrival_month", "market_segment"], observed=True)
    .agg(
        reservas=("market_segment", "size"),
        taxa_cancelamento=("is_canceled", "mean"),
    )
    .reset_index()
)

fig, eixos = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(
    data=mensal_segmento,
    x="arrival_month",
    y="reservas",
    hue="market_segment",
    marker="o",
    ax=eixos[0],
)
eixos[0].set(title="Volume por segmento", xlabel="", ylabel="Reservas")
sns.lineplot(
    data=mensal_segmento,
    x="arrival_month",
    y="taxa_cancelamento",
    hue="market_segment",
    marker="o",
    ax=eixos[1],
)
eixos[1].set(title="Cancelamento por segmento", xlabel="Mês de chegada", ylabel="Taxa")
fig.tight_layout()
salvar(fig, "serie-mensal-por-segmento", mensal_segmento)
mensal_segmento.tail(10)

,arrival_month,market_segment,reservas,taxa_cancelamento
120,2017-07-01,Corporate,131,0.343511
121,2017-07-01,Direct,708,0.185028
122,2017-07-01,Groups,246,0.548780
123,2017-07-01,Offline TA/TO,721,0.255201
124,2017-07-01,Online TA,3484,0.426234
125,2017-08-01,Corporate,60,0.050000
126,2017-08-01,Direct,667,0.200900
127,2017-08-01,Groups,171,0.204678
128,2017-08-01,Offline TA/TO,762,0.212598
129,2017-08-01,Online TA,3252,0.455105


# 4ª etapa · Relações multivariadas

A matriz de Spearman resume associações monotônicas entre as medidas numéricas e o
indicador de cancelamento. Associação não implica causalidade; variáveis posteriores ao
resultado são deliberadamente excluídas.

In [14]:
colunas_correlacao = list(dict.fromkeys(numericas + binary_columns))
correlacao = df[colunas_correlacao].corr(method="spearman")
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    correlacao,
    cmap="vlag",
    center=0,
    annot=len(colunas_correlacao) <= 12,
    fmt=".2f",
    square=True,
    ax=ax,
)
ax.set_title("Associações de Spearman entre variáveis numéricas e binárias")
fig.tight_layout()
salvar(fig, "correlacao-spearman", correlacao.reset_index(names="variavel"))

associacao_cancelamento = (
    correlacao["is_canceled"]
    .drop("is_canceled")
    .sort_values(key=lambda valores: valores.abs(), ascending=False)
    .rename("spearman_com_cancelamento")
    .to_frame()
)
associacao_cancelamento

,spearman_com_cancelamento
lead_time,0.316635
previous_cancellations,0.270233
total_of_special_requests,-0.258520
required_car_parking_spaces,-0.197397
booking_changes,-0.185107
previous_bookings_not_canceled,-0.115354
days_in_waiting_list,0.098237
is_repeated_guest,-0.084793
adults,0.067027
total_guests,0.051380


# 5ª etapa · Valores ausentes

A ausência é quantificada por campo. `agent`, `company`, `country` e `children` podem ter
valores não informados; o relatório não os imputa nem confunde ausência com uma categoria
válida.

In [15]:
ausentes = (
    df.isna()
    .sum()
    .rename("quantidade")
    .to_frame()
    .assign(percentual=lambda tabela: (100 * tabela["quantidade"] / len(df)).round(2))
    .query("quantidade > 0")
    .sort_values("quantidade", ascending=False)
    .reset_index(names="variavel")
)
ausentes.to_csv(TABELAS / "valores-ausentes.csv", index=False, sep=";", encoding="utf-8")

if not ausentes.empty:
    grafico_ausentes = ausentes.head(15).sort_values("percentual")
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(grafico_ausentes))))
    sns.barplot(data=grafico_ausentes, x="percentual", y="variavel", color=BARRA, ax=ax)
    ax.set(title="Campos com maior percentual ausente", xlabel="Reservas ausentes (%)", ylabel="")
    fig.tight_layout()
    salvar(fig, "valores-ausentes", ausentes)

ausentes

,variavel,quantidade,percentual
0,company,112593,94.31
1,agent,16340,13.69
2,country,488,0.41
3,children,4,0.00


# 6ª etapa · Valores extremos

A regra do intervalo interquartil (IQR) sinaliza observações além de `Q1 − 1,5 × IQR` e
`Q3 + 1,5 × IQR`. O relatório quantifica cada campo; os registros não são excluídos, pois
valores extremos podem representar tarifas ou estadias legítimas.

In [16]:
linhas_extremos = []
for coluna in numericas:
    serie = df[coluna].dropna()
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr
    mascara = (serie < inferior) | (serie > superior)
    linhas_extremos.append(
        {
            "variavel": coluna,
            "limite_inferior": inferior,
            "limite_superior": superior,
            "quantidade_extrema": int(mascara.sum()),
            "percentual_extremo": round(100 * mascara.mean(), 2),
            "minimo_observado": serie.min(),
            "maximo_observado": serie.max(),
        }
    )

valores_extremos = pd.DataFrame(linhas_extremos).sort_values("percentual_extremo", ascending=False)
valores_extremos.to_csv(
    TABELAS / "valores-extremos-iqr.csv", index=False, sep=";", encoding="utf-8"
)
valores_extremos

,variavel,limite_inferior,limite_superior,quantidade_extrema,percentual_extremo,minimo_observado,maximo_observado
16,total_guests,2.000,2.000,37339,31.27,0.00,55.0
6,adults,2.000,2.000,29710,24.88,0.00,55.0
11,booking_changes,0.000,0.000,18076,15.14,0.00,21.0
7,children,0.000,0.000,8590,7.20,0.00,10.0
13,required_car_parking_spaces,0.000,0.000,7416,6.21,0.00,8.0
9,previous_cancellations,0.000,0.000,6484,5.43,0.00,26.0
15,total_nights,-1.000,7.000,5257,4.40,0.00,69.0
17,adr,-15.775,211.065,3793,3.18,-6.38,5400.0
12,days_in_waiting_list,0.000,0.000,3698,3.10,0.00,391.0
10,previous_bookings_not_canceled,0.000,0.000,3620,3.03,0.00,72.0


## Relações entre categorias e cancelamento

A taxa de cancelamento é cruzada por tipo de hotel e tipo de depósito. A tabela de contagens
acompanha a taxa para evitar interpretar grupos pequenos como padrões estáveis.

In [17]:
taxa_hotel_deposito = df.pivot_table(
    index="hotel",
    columns="deposit_type",
    values="is_canceled",
    aggfunc="mean",
)
contagem_hotel_deposito = pd.crosstab(df["hotel"], df["deposit_type"])

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(taxa_hotel_deposito, annot=True, fmt=".1%", cmap="YlOrRd", vmin=0, vmax=1, ax=ax)
ax.set(
    title="Taxa de cancelamento por hotel e tipo de depósito",
    xlabel="Tipo de depósito",
    ylabel="Hotel",
)
fig.tight_layout()
salvar(fig, "cancelamento-hotel-deposito", taxa_hotel_deposito.reset_index())

contagem_hotel_deposito

deposit_type,No Deposit,Non Refund,Refundable
hotel,,,
City Hotel,66442,12868,20
Resort Hotel,38199,1719,142


# 7ª etapa · Seleção preliminar de características

Para uma futura tarefa de previsão de cancelamento, `is_canceled` é o alvo. `reservation_status`
e `reservation_status_date` revelam informação posterior ao resultado e devem ser excluídas.
Agência, empresa e país têm alta cardinalidade ou ausência relevante e exigem validação antes
de entrarem em um modelo.

In [18]:
ALVO = "is_canceled"
VAZAMENTO = {"reservation_status", "reservation_status_date"}
ALTA_CARDINALIDADE = {"agent", "company", "country"}


def classificar_decisao(variavel: str) -> str:
    if variavel == ALVO:
        return "alvo"
    if variavel in VAZAMENTO:
        return "excluir por vazamento"
    if variavel in ALTA_CARDINALIDADE:
        return "avaliar cardinalidade/ausência"
    return "candidata"


selecao_preliminar = dicionario[["variavel", "Tipo", "descricao"]].copy()
selecao_preliminar["decisao"] = selecao_preliminar["variavel"].map(classificar_decisao)
selecao_preliminar.to_csv(
    TABELAS / "selecao-preliminar-caracteristicas.csv",
    index=False,
    sep=";",
    encoding="utf-8",
)
selecao_preliminar

,variavel,Tipo,descricao,decisao
0,hotel,Nominal,Tipo de hotel da reserva: City Hotel ou Resort...,candidata
1,is_canceled,Binária,Indicador de cancelamento da reserva (1 = canc...,alvo
2,lead_time,Discreta,Número de dias entre a data de reserva e a dat...,candidata
3,arrival_date_year,Discreta,Ano previsto para a chegada,candidata
4,arrival_date_month,Nominal,Mês previsto para a chegada,candidata
5,arrival_date_week_number,Discreta,Semana do ano prevista para a chegada,candidata
6,arrival_date_day_of_month,Discreta,Dia do mês previsto para a chegada,candidata
7,stays_in_weekend_nights,Discreta,Número de noites de fim de semana (sábado ou d...,candidata
8,stays_in_week_nights,Discreta,Número de noites de semana (segunda a sexta-fe...,candidata
9,adults,Discreta,Número de adultos incluídos na reserva,candidata


# Síntese da exploração

Os indicadores abaixo são calculados da base carregada, sem conclusões pré-escritas. Use-os
junto às tabelas e figuras para orientar a preparação dos dados e a formulação do problema.

In [19]:
taxa_geral = df["is_canceled"].mean()
segmento_principal = df["market_segment"].value_counts().idxmax()
mediana_antecedencia = df["lead_time"].median()
mais_ausente = ausentes.iloc[0]["variavel"] if not ausentes.empty else "nenhuma"

insights = pd.DataFrame(
    [
        {"indicador": "Reservas", "valor": f"{len(df):,}"},
        {"indicador": "Taxa geral de cancelamento", "valor": f"{taxa_geral:.1%}"},
        {"indicador": "Segmento mais frequente", "valor": str(segmento_principal)},
        {"indicador": "Mediana de antecedência (dias)", "valor": f"{mediana_antecedencia:.0f}"},
        {"indicador": "Campo com mais ausências", "valor": str(mais_ausente)},
        {
            "indicador": "Período de chegada",
            "valor": f"{df['arrival_date'].min().date()} a {df['arrival_date'].max().date()}",
        },
    ]
)
insights.to_csv(TABELAS / "sintese.csv", index=False, sep=";", encoding="utf-8")
insights

,indicador,valor
0,Reservas,"119,390"
1,Taxa geral de cancelamento,37.0%
2,Segmento mais frequente,Online TA
3,Mediana de antecedência (dias),69
4,Campo com mais ausências,company
5,Período de chegada,2015-07-01 a 2017-08-31


## Saídas e próximos passos

As tabelas ficam em `reports/tables/eda-exploratoria/` e as figuras em `docs/imagens/eda/`.
A exploração preserva a base bruta. Decisões de imputação, tratamento de extremos e seleção
definitiva de preditores devem ser documentadas na etapa de preparação; em particular,
`reservation_status` e `reservation_status_date` não são preditores válidos para antecipar
cancelamentos.